# MGS-22 : MGS contre mealpy — le bench croisé lib-vs-lib sur le Sudoku

[Jusqu'ici](MGS-6-Benchmarks.ipynb), les bancs de la série comparaient des **composés entre eux** — WOA contre EO contre Islands, tous construits dans le même moteur. Et quand [Sudoku-5](../../Sudoku/Sudoku-5-PSO-Csharp.ipynb) confrontait le solveur maison à GeneticSharp puis à MetaGeneticSharp (Tranches 2-3), chaque moteur tournait *dans son langage*, avec sa propre fonction de coût et son propre budget — la comparaison était honnête en qualité, approximative en méthode.

Ce notebook fait le banc que l'axe 4 de l'issue #11977 demande : **la même expérience, deux bibliothèques, deux langages, dans une seule exécution** — [MetaGeneticSharp](https://github.com/jsboige/MetaGeneticSharp) (C#/.NET 9, DLLs du sous-module) contre [mealpy](https://github.com/thieu1995/mealpy) 3.0.2 (Python/NumPy, la référence du catalogue open-source des métaheuristiques) — reliés par le pont PythonNet dans le kernel .NET. L'hypothèse de départ est celle du user, citée telle quelle : *« possible que les perfs ne suivent pas mealpy sous NumPy optimisé, mais la comparaison sera intéressante »*. Un résultat « MGS plus lent » est donc un livrable valide — à condition qu'il soit **mesuré avec la méthode** : mêmes grilles, mêmes budgets d'évaluations, graines nommées.

## 1. Le protocole apparié, pré-enregistré

Avant d'exécuter quoi que ce soit, le protocole — parce qu'un bench dont les règles sont écrites après les résultats n'est pas un bench.

| Élément | Valeur (fixée d'avance) |
|---|---|
| **Grille** | `Easy[0]` de Sudoku_Easy51 — la même que MGS-21 et les Tranches 1-4 de Sudoku-5 : 36 cellules vides, 45 indices |
| **Représentation** | R1 : vecteur continu $[1,10)^{36}$, décodage par arrondi + clamp — le substrat continu, identique des deux côtés |
| **Fonction de coût** | conflits totaux (lignes + colonnes + blocs) — **la même implémentée deux fois** (C# et Python), égalité vérifiée par sanity check |
| **Moteurs** | PSO canonique à vélocité : composé `ParticleSwarmOptimization` de MetaGeneticSharp contre `OriginalPSO` de mealpy |
| **Budget** | ~8 000 évaluations par course : population 50 × 160 générations/epochs, **évals réellement consommées instrumentées** des deux côtés |
| **Graines** | {0, 1, 7, 42} — nommées, une par course, 4 courses par moteur |
| **Mesures** | (a) qualité : conflits finaux, médiane + min-max sur 4 graines · (b) vitesse : ms par course, **médiane de 3 répétitions par graine** (amendement post-run-pilote, motivé en §2) · (c) coût par évaluation : ms/éval sur 500 évaluations de fitness isolées, médiane de 5 répétitions |

**Critères de lecture, pré-enregistrés.** (1) *Qualité* : un moteur domine si sa médiane de conflits est strictement inférieure ET que ses maxima restent sous les minima de l'autre ; sinon « comparable ». (2) *Vitesse* : rapport des ms/éval moyens. (3) Le verdict global reprend l'hypothèse user — chaque axe est rapporté séparément, **aucune compensation entre axes** (un moteur plus lent ET de même qualité se déclare comme tel).

Pourquoi ce design est le seul honnête : comparer « MGS 37 s contre mealpy 3 s » pris sur deux notebooks, deux fonctions de coût voisines et deux budgets différents ne prouve rien — c'est la comparaison que la Tranche 3 de Sudoku-5 *subissait* (37 033 ms pour 200 000 évaluations, budget et implémentation non appariés). Ici chaque facteur autre que la bibliothèque (grille, coût, budget, graine) est **cloué**.

In [1]:
// === MGS-22 : socle commun — DLLs MGS, grille de référence, fonction de coût ===
// Même socle que MGS-21 : la représentation R1 (continu + arrondi) est le substrat du bench.
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Infrastructure.Framework.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Infrastructure.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Domain.dll"
using MetaGeneticSharp;
using GeneticSharp;
using System.Diagnostics;

// Grille facile Easy[0] de Sudoku_Easy51.txt — la MÊME que MGS-21 et Sudoku-5 Tranches 1-4.
public static string PuzzleLine22 = "902005403100063025508407060026309001057010290090670530240530600705200304080041950";

public static int[,] ParsePuzzle22()
{
    var g = new int[9, 9];
    for (int i = 0; i < 81; i++) g[i / 9, i % 9] = PuzzleLine22[i] - '0';
    return g;
}

// Fonction de coût du bench : conflits totaux (lignes + colonnes + blocs) sur grille PLEINE.
// Renvoie 0 ssi résolu. Le côté Python réimplémente exactement ce comptage.
public static int CountConflicts22(int[,] g)
{
    int conflicts = 0;
    for (int i = 0; i < 9; i++)
    {
        var row = new HashSet<int>(); var col = new HashSet<int>(); var blk = new HashSet<int>();
        for (int j = 0; j < 9; j++)
        {
            if (!row.Add(g[i, j])) conflicts++;
            if (!col.Add(g[j, i])) conflicts++;
            int br = 3 * (i / 3) + j / 3, bc = 3 * (i % 3) + j % 3;
            if (!blk.Add(g[br, bc])) conflicts++;
        }
    }
    return conflicts;
}

public static int CountEmpty22(int[,] p) { int n = 0; foreach (var v in p) if (v == 0) n++; return n; }

public static List<(int r, int c)> EmptyCells22(int[,] p)
{
    var l = new List<(int, int)>();
    for (int r = 0; r < 9; r++) for (int c = 0; c < 9; c++) if (p[r, c] == 0) l.Add((r, c));
    return l;
}

// Décodage R1 : arrondi + clamp vers 1..9 sur les cellules vides, ordre de lecture.
public static int[,] DecodeR1_22(double[] genes)
{
    var Puzzle = ParsePuzzle22();
    var empties = EmptyCells22(Puzzle);
    var g = (int[,])Puzzle.Clone();
    for (int k = 0; k < empties.Count; k++)
        g[empties[k].r, empties[k].c] = Math.Max(1, Math.Min(9, (int)Math.Round(genes[k])));
    return g;
}

var Puzzle22 = ParsePuzzle22();
Console.WriteLine($"Grille de référence : {CountEmpty22(Puzzle22)} cellules vides, " +
                  $"{81 - CountEmpty22(Puzzle22)} indices fixes, {EmptyCells22(Puzzle22).Count} gènes R1.");

Grille de référence : 36 cellules vides, 45 indices fixes, 36 gènes R1.


### Interprétation : le socle est posé

Rien de neuf côté C# — c'est voulu. Le chromosome R1 à 36 gènes continus et le compte de conflits totaux sont exactement ceux de MGS-21 : le présent bench **hérite du substrat validé** plutôt que d'en réintroduire un. La seule pièce nouvelle est `DecodeR1_22` découplé du chromosome : le décodage vit comme une fonction pure sur un vecteur, parce que le côté Python devra appliquer **exactement la même transformation** aux solutions mealpy.

Un détail d'implémentation à ne pas laisser passer : `(int)Math.Round` en C# applique l'arrondi bancaire (les demi-entiers vont au pair), et `int(round(...))` en Python aussi — mais ce n'est pas une raison pour *croire* les deux paysages identiques : sur des gènes tirés uniformément dans $[1,10)$, la probabilité de tomber exactement sur un demi-entier est nulle en théorie et négligeable en pratique. L'argument, pourtant, ne sera **pas invoqué** : la sanity check de la section suivante tranche sur des données réelles — trois vecteurs témoins coûtés indépendamment des deux côtés.

In [2]:
// === Moteur MGS : chromosome R1, fitness instrumentée, PSO canonique composé ===
// Le PSO canonique à vélocité (axe 1 de #11977) : récurrence w/c1/c2 de Shi & Eberhart.
public class SudokuR1Chromosome22 : ChromosomeBase
{
    private const double LO = 1.0, HI = 10.0;
    public SudokuR1Chromosome22() : base(EmptyCells22(ParsePuzzle22()).Count) { CreateGenes(); }
    public override Gene GenerateGene(int index)
        => new Gene(RandomizationProvider.Current.GetDouble(LO, HI));
    public override IChromosome CreateNew() => new SudokuR1Chromosome22();
    public double[] ToGenes() { var v = new double[Length]; for (int i = 0; i < Length; i++) v[i] = (double)GetGene(i).Value; return v; }
    public int[,] ToGrid() => DecodeR1_22(ToGenes());
}

// Fitness instrumentée : chaque évaluation est comptée — le budget se mesure, il ne se suppose pas.
public class SudokuR1Fitness22 : IFitness
{
    public static int Evals;
    public double Evaluate(IChromosome chromosome)
    {
        Evals++;
        return -CountConflicts22(((SudokuR1Chromosome22)chromosome).ToGrid());
    }
}

public static class Mgs22Host
{
    public static (int conflicts, int evals, double ms, double[] genes) RunPso(int seed, int popSize, int maxGens)
    {
        // Seeding AVANT création de population : le RNG est consommé par CreateNew()
        // de chaque individu initial (leçon #12071 / MGS-21).
        FastRandomRandomization.ResetSeed(seed);
        var compound = MetaHeuristicsService.CreateMetaHeuristicByName(
            "ParticleSwarmOptimization", maxGens, popSize);
        var adam = new SudokuR1Chromosome22();
        var pop = new MetaPopulation(popSize, popSize, adam);
        var ga = new MetaGeneticAlgorithm(
            pop, new SudokuR1Fitness22(),
            new EliteSelection(), new UniformCrossover(0.5f), new UniformMutation(true),
            compound);
        ga.Termination = new GenerationNumberTermination(maxGens);
        SudokuR1Fitness22.Evals = 0;
        var sw = Stopwatch.StartNew();
        ga.Start();
        sw.Stop();
        var best = (SudokuR1Chromosome22)ga.BestChromosome;
        return (CountConflicts22(best.ToGrid()), SudokuR1Fitness22.Evals,
                sw.Elapsed.TotalMilliseconds, best.ToGenes());
    }
}

// Échauffement JIT (course jetée), puis course témoin graine 7.
var warmupMgs = Mgs22Host.RunPso(123, 50, 10);
var demoMgs = Mgs22Host.RunPso(7, 50, 160);
Console.WriteLine($"MGS PSO (graine 7, témoin) : {demoMgs.Item1} conflits, " +
                  $"{demoMgs.Item2} évaluations, {demoMgs.Item3:F0} ms.");

MGS PSO (graine 7, témoin) : 46 conflits, 8000 évaluations, 1134 ms.


### Interprétation : ce que le moteur MGS expose

Trois choix de wiring portent la validité du bench :

1. **Le compteur d'évaluations vit dans la fitness, pas dans un estimateur.** `SudokuR1Fitness22.Evals` s'incrémente à chaque `Evaluate` — le « budget ~8 000 » du protocole se **vérifie** course par course au lieu d'être supposé depuis `population × générations`. C'est la même instrumentation que MGS-21, et elle rend la comparaison des budgets lisible dans le tableau final plutôt qu'arguée dans la prose.
2. **Le seeding précède la création de population.** `FastRandomRandomization.ResetSeed(seed)` avant `new SudokuR1Chromosome22()` — le RNG est consommé par `CreateNew()` de chaque individu initial ; re-seeder après ne fixe rien (leçon #12071). Les graines {0, 1, 7, 42} produisent des courses **reproductibles** : ré-exécuter ce notebook redonne les mêmes conflits.
3. **L'échauffement est jeté, pas compté.** La première course (graine 123, 10 générations) paie la compilation JIT et l'amorçage des DLLs ; la course témoin et le bench mesurent du code chaud. Le côté mealpy aura son échauffement symétrique — sinon la comparaison des temps mesurerait l'amorçage d'un côté et le régime permanent de l'autre.

La course témoin (graine 7) donne le premier point de données. Regardez ses conflits : de l'ordre de la colonne R1/PSO de MGS-21 (médiane 45,0 à budget égal), pas de la résolution. C'est attendu — le PSO continu plafonne sur le Sudoku discret, et il plafonnera **des deux côtés** de la comparaison à venir : c'est précisément pourquoi le protocole compare à budget égal plutôt qu'à résolution.

In [3]:
// === Le pont PythonNet : mealpy dans le même kernel, la même exécution ===
// Recette validée (probe SC-4) : pythonnet 3.1.0 + DLL autonome du Python python.org.
// Le Python 3.13 du système porte mealpy 3.0.2 (version affichée ci-dessous = preuve).
#r "nuget: pythonnet,3.1.0"
using Python.Runtime;
Runtime.PythonDLL = @"C:\Users\jsboi\AppData\Local\Programs\Python\Python313\python313.dll";
PythonEngine.Initialize();

// Le problème Python : decode + coût réimplémentés à l'identique, compteur d'évals,
// solveur mealpy avec seed EXPLICITE en solve() (API 3.x — le seed du constructeur est
// ignoré, vérifié en amont) et journal muet (log_to='nothing').
public static PyModule S22;
using (Py.GIL())
{
    S22 = Py.CreateScope();
    S22.Set("puzzle_line22", PuzzleLine22);
    S22.Exec(@"import sys
import mealpy
from mealpy.swarm_based.PSO import OriginalPSO
from mealpy import Problem, FloatVar
import json as _json

puzzle = [int(ch) for ch in puzzle_line22]
empties = [(i // 9, i % 9) for i in range(81) if puzzle[i] == 0]

def decode(vec):
    g = [puzzle[r * 9:(r + 1) * 9] for r in range(9)]
    for k in range(len(empties)):
        r, c = empties[k]
        v = int(round(float(vec[k])))
        g[r][c] = max(1, min(9, v))
    return g

def cost(g):
    conflicts = 0
    for i in range(9):
        units = ([g[i][j] for j in range(9)],
                 [g[j][i] for j in range(9)],
                 [g[3 * (i // 3) + j // 3][3 * (i % 3) + j % 3] for j in range(9)])
        for unit in units:
            seen = set()
            for v in unit:
                if v in seen:
                    conflicts += 1
                seen.add(v)
    return conflicts

def cost_of_vector(vec):
    return cost(decode(vec))

PY_EVALS = [0]

class SudokuProblem(Problem):
    def __init__(self, bounds=None, minmax='min', **kwargs):
        super().__init__(bounds, minmax, log_to='nothing', **kwargs)
    def obj_func(self, x):
        PY_EVALS[0] += 1
        return float(cost(decode(x)))

def run_mealpy_pso(seed, pop_size, epoch):
    import time
    PY_EVALS[0] = 0
    prob = SudokuProblem(bounds=FloatVar(lb=(1.0,) * len(empties), ub=(10.0,) * len(empties), name='genes'), minmax='min')
    model = OriginalPSO(epoch=epoch, pop_size=pop_size)
    t0 = time.perf_counter()
    g_best = model.solve(prob, seed=seed)
    dt = (time.perf_counter() - t0) * 1000.0
    sol = _json.dumps([float(v) for v in g_best.solution])
    return cost(decode(g_best.solution)), PY_EVALS[0], dt, sol

def bench_mealpy(seeds_json, pop_size, epoch, reps=3):
    out = []
    for sd in _json.loads(seeds_json):
        runs = [run_mealpy_pso(sd, pop_size, epoch) for _ in range(reps)]
        cs = [r[0] for r in runs]
        es = [r[1] for r in runs]
        ts = sorted(r[2] for r in runs)
        med = ts[len(ts) // 2] if len(ts) % 2 == 1 else (ts[len(ts) // 2 - 1] + ts[len(ts) // 2]) / 2.0
        out.append({'seed': sd, 'conflicts': cs[0], 'all_same': len(set(cs)) == 1,
                    'evals': es[0], 'ms': med, 'sol': runs[0][3]})
    return _json.dumps(out)

def time_python_evals(vecs_json, reps=5):
    import time
    vecs = _json.loads(vecs_json)
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        for v in vecs:
            cost_of_vector(v)
        ts.append((time.perf_counter() - t0) * 1000.0)
    ts.sort()
    return ts[len(ts) // 2]

__mealpy_ver__ = 'mealpy ' + mealpy.__version__ + ' sur Python ' + sys.version.split()[0]");
    Console.WriteLine($"Pont PythonNet actif : {S22.Get<string>("__mealpy_ver__")}");
}

// --- Sanity check : la fonction de coût est-elle la MÊME des deux côtés ? ---
// 3 vecteurs témoins DÉTERMINISTES (LCG écrit à la main), décodés et costés des deux côtés.
public static double[] LcgVector22(int seed, int n)
{
    uint state = (uint)seed;
    var v = new double[n];
    for (int i = 0; i < n; i++)
    {
        state = state * 1664525u + 1013904223u;
        v[i] = 1.0 + (state / 4294967296.0) * 9.0; // uniforme dans [1, 10)
    }
    return v;
}

var witnessVectors = new[] { LcgVector22(1, 36), LcgVector22(2, 36), LcgVector22(3, 36) };
using (Py.GIL())
{
    S22.Set("__witness_json__",
        System.Text.Json.JsonSerializer.Serialize(witnessVectors.Select(v => v.ToList()).ToList()));
    S22.Exec(@"__py_costs_json__ = _json.dumps([cost_of_vector(v) for v in _json.loads(__witness_json__)])");
    var pyCosts = System.Text.Json.JsonSerializer.Deserialize<List<int>>(S22.Get<string>("__py_costs_json__"));
    bool allEqual = true;
    for (int i = 0; i < witnessVectors.Length; i++)
    {
        int csCost = CountConflicts22(DecodeR1_22(witnessVectors[i]));
        bool eq = csCost == pyCosts[i];
        allEqual &= eq;
        Console.WriteLine($"  vecteur témoin {i + 1} : coût C# = {csCost}, coût Python = {pyCosts[i]} -> {(eq ? "IDENTIQUE" : "DIFFERENT")}");
    }
    Console.WriteLine(allEqual
        ? "Sanity check PASSE : la fonction de coût est identique des deux côtés -- le bench est valide."
        : "Sanity check ECHOUE : les fonctions de coût diffèrent -- le bench serait invalide.");
}

Installing Packages pythonnet

Pont PythonNet actif : mealpy 3.0.2 sur Python 3.13.7


  vecteur témoin 1 : coût C# = 67, coût Python = 67 -> IDENTIQUE


  vecteur témoin 2 : coût C# = 71, coût Python = 71 -> IDENTIQUE


  vecteur témoin 3 : coût C# = 60, coût Python = 60 -> IDENTIQUE


Sanity check PASSE : la fonction de coût est identique des deux côtés -- le bench est valide.


### Interprétation : pourquoi la sanity check porte tout le bench

Le point fragile d'une comparaison cross-langage n'est pas le solveur — c'est la **fonction de coût**. Si le C# comptait les doublons avec une convention différente du Python, les deux moteurs n'optimiseraient pas le même paysage, et chaque milliseconde comparée serait du bruit raffiné. D'où le protocole en deux temps :

- **des vecteurs témoins déterministes** — générés par un LCG écrit à la main ($state \leftarrow state \times 1664525 + 1013904223$, la paire classique de Numerical Recipes), parce que `System.Random` et le RNG de NumPy ne produisent pas les mêmes suites : le LCG garantit que les deux côtés coûtent **exactement les mêmes points** de l'espace ;
- **le coût calculé indépendamment de chaque côté**, puis confronté — trois vecteurs, trois paires.

Si les trois paires coïncident, l'égalité des fonctions de coût est établie *sur données* — et l'argument théorique sur les arrondis de demi-entiers devient superflu.

Le pont lui-même suit la recette du probe : `Runtime.PythonDLL` pointé sur la DLL autonome du Python python.org 3.13, code Python injecté par `scope.Exec` en chaîne verbatim, données échangées en JSON (`Set` pour descendre, `Get` pour remonter) — jamais par lecture directe de variables Python complexes, qui ne traversent pas proprement la frontière. Le scope `S22` est gardé en champ statique : les cellules suivantes (course témoin mealpy, bench complet, profil de coût) réutilisent le même espace de noms Python et les mêmes définitions. Notez aussi les **graines des deux moteurs passées explicitement** : `solve(prob, seed=N)` côté mealpy (le paramètre du constructeur est silencieusement ignoré par l'API 3.x — un piège documenté dans le code), `ResetSeed(N)` côté MGS.

In [4]:
// === Moteur mealpy : course témoin + contre-vérification croisée du vainqueur ===
using (Py.GIL())
{
    // Échauffement symétrique (course jetée), puis course témoin graine 7.
    S22.Exec(@"_wu_c, _wu_e, _wu_t, _wu_sol = run_mealpy_pso(123, 50, 10)
__d_c__, __d_e__, __d_t__, __d_sol__ = run_mealpy_pso(7, 50, 160)");
    int dConflicts = S22.Get<int>("__d_c__");
    int dEvals = S22.Get<int>("__d_e__");
    double dMs = S22.Get<double>("__d_t__");
    Console.WriteLine($"mealpy OriginalPSO (graine 7, témoin) : {dConflicts} conflits, " +
                      $"{dEvals} évaluations, {dMs:F0} ms.");

    // Contre-vérification croisée : le vainqueur mealpy, décodé et costé côté C#.
    var solJson = S22.Get<string>("__d_sol__");
    var genes = System.Text.Json.JsonSerializer.Deserialize<double[]>(solJson);
    int csRecheck = CountConflicts22(DecodeR1_22(genes));
    Console.WriteLine($"Contre-vérif croisée : coût C# du meilleur mealpy = {csRecheck} " +
                      $"(Python rapporte {dConflicts}) -> {(csRecheck == dConflicts ? "IDENTIQUE" : "DIFFERENT")}");
}

mealpy OriginalPSO (graine 7, témoin) : 26 conflits, 8050 évaluations, 751 ms.


Contre-vérif croisée : coût C# du meilleur mealpy = 26 (Python rapporte 26) -> IDENTIQUE


## 2. Le croisement — 2 moteurs × 4 graines à budget égal

Le plan est maintenant entièrement câblé : même grille, même coût (prouvé sur données), même représentation. Reste à exécuter le plan de la section 1 — population 50, 160 générations/epochs, graines {0, 1, 7, 42} — et à rapporter **les quatre colonnes** : conflits finaux, évaluations réellement consommées, temps, et le ratio ms/éval. Les courses mealpy tournent en une seule entrée dans le scope Python (la boucle vit côté Python, les résultats reviennent en JSON) ; les courses MGS tournent côté C#. Médianes et min-max se calculent sur les 4 graines de chaque moteur.

**Amendement de protocole, motivé par le run pilote.** La première exécution complète a montré le défaut du timing single-run : les temps MGS varient de ±30 % d'une exécution à l'autre (paliers de garbage collection .NET — un run a mesuré 1 132 ms là où un autre donne 660 ms pour la même course seedée), soit *plus que l'écart entre moteurs*. Un protocole vitesse qui bouge plus que la grandeur mesurée ne mesure rien. Amendement, appliqué symétriquement : **3 répétitions par graine et par moteur, la colonne ms rapporte la médiane** des 3 ; les conflits seedés doivent être identiques sur les 3 répétitions (et le notebook l'affiche — preuve de déterminisme en direct, 12 courses par moteur). La fitness isolée (section 3) suit la même règle à 5 répétitions. La qualité, elle, ne bouge pas : elle est seedée, le pilote l'a montré chiffre par chiffre.

Rappel des critères pré-enregistrés : *qualité* — médiane strictement inférieure ET max sous min de l'autre, sinon « comparable » ; *vitesse* — rapport des ms/éval moyens ; *aucune compensation* entre les axes. Le verdict sur l'hypothèse user (« possible que les perfs ne suivent pas mealpy ») se lira sur l'axe vitesse ; l'axe qualité dira si les deux implémentations du PSO canonique convergent pareil à budget égal.

In [5]:
// === LE BENCH : 2 moteurs x 4 graines {0,1,7,42}, population 50, 160 générations ===
public class BenchRow22
{
    public int seed { get; set; }
    public int conflicts { get; set; }
    public bool all_same { get; set; }
    public int evals { get; set; }
    public double ms { get; set; }
    public string sol { get; set; }
}

int[] Seeds22 = { 0, 1, 7, 42 };

// --- Côté MGS (C#) : 3 répétitions par graine, ms = médiane (amendement §2) ---
var mgsRows = new List<(int seed, int conflicts, int evals, double ms, bool allSame)>();
foreach (var sd in Seeds22)
{
    var runs3 = new List<(int c, int e, double t)>();
    for (int rep = 0; rep < 3; rep++)
    {
        var r = Mgs22Host.RunPso(sd, 50, 160);
        runs3.Add((r.Item1, r.Item2, r.Item3));
    }
    var times = runs3.Select(x => x.t).OrderBy(t => t).ToList();
    double med = times[1];
    mgsRows.Add((sd, runs3[0].c, runs3[0].e, med, runs3.All(x => x.c == runs3[0].c)));
}

// --- Côté mealpy (Python, boucle unique dans le scope) ---
string mealpyJson;
using (Py.GIL())
{
    S22.Set("__seeds_json__", System.Text.Json.JsonSerializer.Serialize(Seeds22.ToList()));
    S22.Exec(@"__bench_json__ = bench_mealpy(__seeds_json__, 50, 160)");
    mealpyJson = S22.Get<string>("__bench_json__");
}
var mealpyRows = System.Text.Json.JsonSerializer.Deserialize<List<BenchRow22>>(mealpyJson);

// --- Table ---
static double Median22(List<int> xs)
{
    var s = xs.OrderBy(x => x).ToList();
    return (s.Count % 2 == 1) ? s[s.Count / 2] : (s[s.Count / 2 - 1] + s[s.Count / 2]) / 2.0;
}

Console.WriteLine($"{"moteur",-9} {"graine",6} {"conflits",9} {"evals",7} {"ms",8} {"ms/eval",8}");
foreach (var r in mgsRows)
    Console.WriteLine($"{"MGS",-9} {r.seed,6} {r.conflicts,9} {r.evals,7} {r.ms,8:F0} {r.ms / r.evals,8:F3}");
foreach (var r in mealpyRows)
    Console.WriteLine($"{"mealpy",-9} {r.seed,6} {r.conflicts,9} {r.evals,7} {r.ms,8:F0} {r.ms / r.evals,8:F3}");

var mgsC = mgsRows.Select(r => r.conflicts).ToList();
var mpC = mealpyRows.Select(r => r.conflicts).ToList();
double mgsMsEval = mgsRows.Average(r => r.ms / r.evals);
double mpMsEval = mealpyRows.Average(r => r.ms / r.evals);
Console.WriteLine();
Console.WriteLine($"MGS    : médiane conflits {Median22(mgsC):F1} (min {mgsC.Min()}, max {mgsC.Max()}), ms/éval moyen {mgsMsEval:F3}");
Console.WriteLine($"mealpy : médiane conflits {Median22(mpC):F1} (min {mpC.Min()}, max {mpC.Max()}), ms/éval moyen {mpMsEval:F3}");
Console.WriteLine($"Rapport ms/éval mealpy/MGS : {mpMsEval / mgsMsEval:F2}x");
int detMgs = mgsRows.Count(r => r.allSame) + mealpyRows.Count(r => r.all_same);
Console.WriteLine($"Déterminisme : conflits identiques sur les 3 répétitions pour {detMgs}/8 paires graine-moteur.");

moteur    graine  conflits   evals       ms  ms/eval


MGS            0        39    8000      857    0,107


MGS            1        41    8000      689    0,086


MGS            7        46    8000      680    0,085


MGS           42        48    8000      692    0,087


...

mealpy        42        31    8050      710    0,088


MGS    : médiane conflits 43,5 (min 39, max 48), ms/éval moyen 0,091


mealpy : médiane conflits 28,5 (min 26, max 31), ms/éval moyen 0,090


Rapport ms/éval mealpy/MGS : 0,99x


Déterminisme : conflits identiques sur les 3 répétitions pour 8/8 paires graine-moteur.


### Lecture du croisement : ex æquo en vitesse, mealpy domine en qualité

Les critères pré-enregistrés, appliqués aux médianes de 3 répétitions :

- **Qualité** : mealpy médiane 28,5 contre 43,5 pour MGS — et **séparation totale des gammes** : le pire mealpy (31 conflits, graine 42) fait mieux que le meilleur MGS (39 conflits, graine 0). Le critère « médiane strictement inférieure ET max sous min » est satisfait dans les deux clauses : domination complète, sans chevauchement de distributions.
- **Vitesse** : rapport ms/éval de **0,99×** — ex æquo statistique. Les médianes par graine se tiennent dans une fourchette serrée côté mealpy (710-735 ms pour ~8 000 évaluations) ; côté MGS (680-692 ms, sauf la graine 0 à 857) la sensibilité GC survit partiellement à la médiane de 3 — deux des trois répétitions de la graine 0 ont pris le palier, preuve que trois répétitions bornent le bruit sans l'éliminer. Et l'amendement de protocole a été décisif ici : le run pilote en timing single-run concluant « 0,8×, mealpy 20 % plus vite » était un **artefact** — un pic de garbage collection .NET (1 132 ms mesurés pour une course qui en prend ~700 à chaud) avait gonflé la moyenne MGS. À médiane de répétitions, l'écart s'évanouit.
- **Déterminisme** : 8/8 paires graine-moteur avec conflits identiques sur les 3 répétitions — le seeding est prouvé *en direct*, et corollaire important : la différence de qualité entre moteurs (39-48 contre 26-31) **n'est pas du bruit**, chaque course seedée reproduit exactement son résultat.

L'hypothèse de départ — *« possible que les perfs ne suivent pas mealpy sous NumPy optimisé »* — se lit donc en deux moitiés : **réfutée sur l'axe vitesse** (MGS tient le rythme de mealpy à budget égal), et **vérifiée en sens inverse sur l'axe qualité** (mealpy devant, nettement). La nuance honnête sur l'axe qualité : les deux « PSO canoniques » ne partagent pas leurs paramètres par défaut (coefficients d'inertie et d'attraction réglés différemment par chaque bibliothèque) — l'axe mesure *la bibliothèque telle qu'on la lance*, ce que mesure un utilisateur réel, pas une course de formules identiques. La section suivante explique pourquoi l'ex æquo vitesse est, malgré tout, le résultat le plus instructif du notebook.

In [6]:
// === Coût par évaluation : la fitness seule, hors moteur, 500 vecteurs identiques ===
// Les vecteurs sont générés côté C# (LCG, graines 42..541) et passés en JSON au Python :
// les DEUX côtés chronomètrent decode+coût sur exactement les mêmes 500 points.
int K22 = 500;
var benchVecs = new List<double[]>();
for (int i = 0; i < K22; i++) benchVecs.Add(LcgVector22(42 + i, 36));

var csTimes = new List<double>();
for (int rep = 0; rep < 5; rep++)
{
    var swRep = Stopwatch.StartNew();
    foreach (var v in benchVecs) CountConflicts22(DecodeR1_22(v));
    swRep.Stop();
    csTimes.Add(swRep.Elapsed.TotalMilliseconds);
}
csTimes.Sort();
double csMs = csTimes[2]; // médiane de 5 (amendement §2)

double pyMs;
using (Py.GIL())
{
    S22.Set("__vecs_json__", System.Text.Json.JsonSerializer.Serialize(benchVecs.Select(v => v.ToList()).ToList()));
    S22.Exec(@"__py_ms__ = time_python_evals(__vecs_json__)");
    pyMs = S22.Get<double>("__py_ms__");
}

Console.WriteLine($"Fitness seule, {K22} vecteurs identiques (médiane de 5 répétitions par côté) :");
Console.WriteLine($"  C#     : {csMs:F1} ms total -> {csMs / K22:F3} ms/éval");
Console.WriteLine($"  Python : {pyMs:F1} ms total -> {pyMs / K22:F3} ms/éval");
Console.WriteLine($"  rapport Python/C# : {pyMs / csMs:F2}x");

Fitness seule, 500 vecteurs identiques (médiane de 5 répétitions par côté) :


  C#     : 3,7 ms total -> 0,007 ms/éval


  Python : 21,3 ms total -> 0,043 ms/éval


  rapport Python/C# : 5,72x


### Lecture du coût par évaluation : fitness C# 5,5× plus rapide — et un bench ex æquo quand même

C'est la cellule qui explique le paradoxe apparent du croisement. Mesurée seule, hors moteur, sur les 500 mêmes vecteurs (médiane de 5 répétitions) :

- **fitness C# : 0,007 ms/éval** (3,7 ms pour 500) — **fitness Python : 0,043 ms/éval** (21,3 ms) : rapport **5,72×** en faveur du C#. L'écart attendu entre code JIT .NET et code objet Python interprété est bien là, massif et stable (5,5-5,7× d'une exécution à l'autre).

Pourtant le bench complet donne 0,091 ms/éval pour MGS contre 0,090 pour mealpy — l'ex æquo presque parfait. La décomposition arithmétique est la leçon du notebook :

| | fitness (mesurée) | moteur (bench − fitness) | total |
|---|---|---|---|
| **MGS** | 0,007 ms/éval | **0,084 ms/éval** | 0,091 |
| **mealpy** | 0,043 ms/éval | **0,047 ms/éval** | 0,090 |

Deux effets opposés s'annulent : la fitness C# est 5,5× plus rapide, mais la **mécanique de génération** de MetaGeneticSharp (sélection, croisement, mutation, structures de population) dépense ~0,084 ms par évaluation, soit **~1,8×** celle de mealpy (~0,047 ms) — et la fitness ne pèse que 8 % du coût total côté MGS. Trois moralités mesurées : *le coût par évaluation d'un moteur n'est pas le coût de sa fitness* ; *un avantage fitness spectaculaire (5,5×) peut être invisible dans le temps total* ; et *un timing single-run aurait conclu n'importe quoi* — 0,8× sur un run, 1,0× sur l'autre, pour le même code.

## Exercice 1 : la revanche du GA — `BaseGA` mealpy contre le composé `Default` MGS

Le croisement ci-dessus compare deux implémentations du **même** algorithme (PSO canonique). L'exercice symétrique naturel est le **GA** : `mealpy.evolutionary_based.GA.BaseGA` contre le composé `"Default"` de MetaGeneticSharp (celui de la colonne R1/GA de MGS-21, médiane 10,5 à 8 000 évaluations). Consignez pour chaque moteur et chaque graine de {0, 1, 7, 42} : conflits finaux, évaluations consommées, temps — puis répondez : *est-ce que l'ordre observé PSO-contre-PSO se retrouve GA-contre-GA, ou est-ce une propriété du couple algorithme × bibliothèque ?*

Plan suggéré : réutiliser `run_mealpy_pso` comme gabarit pour écrire `run_mealpy_ga` (seule la classe du modèle change), et `Mgs22Host.RunPso` pour un `RunGa` (le composé `"Default"` se crée par le même `CreateMetaHeuristicByName`).

In [7]:
// EXERCICE 1 : bench croisé GA mealpy (BaseGA) contre GA MGS (composé "Default")
// Graine par graine, budget ~8000 évals, mêmes colonnes que le croisement PSO.
// Indice : côté Python -> from mealpy.evolutionary_based.GA import BaseGA puis
//          un run_mealpy_ga sur le même SudokuProblem ; côté C# -> composé "Default".
// Etape 1 : implémenter run_mealpy_ga dans le scope S22.
// Etape 2 : implémenter Mgs22Host.RunGa (copie de RunPso, composé "Default").
// Etape 3 : boucler sur les 4 graines des deux côtés, imprimer la table, médianes.
var resultExo1 = null as List<string>;
Console.WriteLine("Exercice a completer : bench croise GA mealpy vs GA MGS (4 graines, budget egal).");

Exercice a completer : bench croise GA mealpy vs GA MGS (4 graines, budget egal).


## Exercice 2 : budget ×4 — l'écart de qualité se referme-t-il ?

Le croisement se conclut (voir la Lecture) à ~8 000 évaluations, budget où la représentation R1 plafonne loin de la résolution. Multipliez le budget par 4 (population 50 × 640 générations) **des deux côtés**, graines {0, 1, 7, 42}, et mesurez : les médianes de conflits descendent-elles de concert ? Le rapport ms/éval bouge-t-il, ou est-il une propriété de la fitness, indépendante du budget ? Un moteur profite-t-il du budget long plus vite que l'autre — et si oui, laquelle des deux implémentations (paramètres par défaut différents : coefficients de Clerc contre Shi & Eberhart, gestion du voisinage) est la cause plausible ?

In [8]:
// EXERCICE 2 : budget x4 (pop 50, 640 générations/epochs), 4 graines, deux moteurs.
// Indice : un seul changement de paramètre par côté -- 160 -> 640.
// Etape 1 : relancer le plan du croisement avec gens/epoch = 640.
// Etape 2 : comparer médianes/min-max à 8000 vs 32000 évals par moteur.
// Etape 3 : rapporter si le rapport ms/éval reste stable (fitness-déterminé) ou dérive.
var resultExo2 = null as List<string>;
Console.WriteLine("Exercice a completer : budget x4 des deux cotes, comparaison des medianes et du rapport ms/eval.");

Exercice a completer : budget x4 des deux cotes, comparaison des medianes et du rapport ms/eval.


## Exercice 3 : profiler la fitness — où va la milliseconde ?

La cellule du coût par évaluation mesure `decode + cost` en bloc. Décomposez : chronométrez séparément (a) le **décodage** (construction de la grille depuis le vecteur), (b) le **compte de conflits** (parcours des 27 unités), sur les 500 mêmes vecteurs, des deux côtés. Le rapport Python/C# est-il homogène entre (a) et (b), ou une des deux étapes porte-t-elle l'écart ? Puis la question qui ferme la boucle : *combien de la ms/éval du bench complet est de la fitness, combien du moteur ?* (différence bench-complet moins fitness-isolée, par côté).

In [9]:
// EXERCICE 3 : profil decode vs cost, 500 vecteurs, deux côtés, puis séparation
// fitness/moteur dans la ms/éval du bench complet.
// Indice : côté C# deux Stopwatches (DecodeR1_22 seul, CountConflicts22 seul) ;
//          côté Python deux time.perf_counter autour de decode(v) et cost(g).
// Etape 1 : mesurer (a) decode seul, (b) cost seul, chaque côté.
// Etape 2 : répartir la ms/éval du bench entre fitness et moteur.
var resultExo3 = null as List<string>;
Console.WriteLine("Exercice a completer : profil decode/cost par cote, repartition fitness vs moteur.");

Exercice a completer : profil decode/cost par cote, repartition fitness vs moteur.


## Résumé et perspectives

**Réponse à l'hypothèse de départ.** Sur ce plan (PSO canonique contre OriginalPSO, grille Easy[0], coût prouvé identique, ~8 000 évaluations, graines {0, 1, 7, 42}, médianes de répétitions) : **ex æquo en vitesse** (0,99× le coût par évaluation), **mealpy domine en qualité** (médiane 28,5 contre 43,5 conflits, séparation totale des gammes 26-31 contre 39-48). L'hypothèse « les perfs ne suivront pas mealpy » est réfutée sur l'axe vitesse — MetaGeneticSharp tient le rythme — et l'écart réel entre bibliothèques se joue sur les **défauts du solveur** (qualité à budget égal), pas sur la vitesse d'exécution.

**Le mécanisme, mesuré.** La fitness C# est 5,72× plus rapide (0,007 contre 0,043 ms/éval), mais la mécanique MGS coûte ~1,8× plus par évaluation (0,084 contre 0,047 ms) — les deux effets se compensent presque exactement (0,091 contre 0,090 ms/éval au total). C'est la démonstration la plus utile du notebook pour la suite de la série : optimiser la fitness C# (déjà marginale à 8 % du total) ne rapprochera MGS de personne ; la cible est la mécanique de génération.

**La méthode, réutilisable.** Sanity check sur vecteurs témoins LCG avant tout bench cross-langage ; seed explicite en `solve()` côté mealpy (le constructeur l'ignore silencieusement — piège documenté ici pour la série) ; `ResetSeed` avant création de population côté MGS ; **tout timing en médiane de répétitions** — l'amendement de ce notebook (un pic GC .NET peut fausser un single-run de ±60 %) vaut pour tous les bancs à venir.

**Perspectives.** (1) Les exercices testent la robustesse du verdict : GA contre GA (Exercice 1 — mealpy a aussi `BaseGA`, MGS son composé `Default`), budget ×4 (Exercice 2). (2) Le point d'entrée pour dépasser mealpy en qualité n'est pas la vitesse brute mais les **composés spécialisés** de la série (seeding MGS-4/MGS-7, îles) : le composé PSO canonique, pris isolément, perd contre les défauts bien réglés de mealpy sur ce cas. (3) Le profiling de l'Exercice 3 dira où va la milliseconde moteur — les 0,084 ms/éval de mécanique MGS étant la cible évidente.